# Лабораторна робота 10. Завдання 4. Мультикласова класифікація: Rock-Paper-Scissors

Виконав: Бабин Богдан Олегович, група 443-4. Базовий ноутбук з методички (`Example_1_multi_class_classifier`) — без змін у навчанні.
Додано: збереження моделі, розпізнавання тестових зображень з набору `rps-validation` (лекція), реальних фото рук з інтернету
та, як додатковий експеримент, фото справжніх каменів, ножиць і паперу.

*Виправлення:* у базовій комірці розпізнавання зображення не нормалізуються (`/255`), хоча мережа навчалась на нормалізованих
даних — тут додано `x / 255.` (без цього всі передбачення спотворені).

# Multi-class Classifier
In this example, you will look at how to build a model to distinguish between more than two classes.

In [ ]:
# Download the train set
!wget -q https://storage.googleapis.com/learning-datasets/rps.zip
# Download the test set
!wget -q https://storage.googleapis.com/learning-datasets/rps-test-set.zip

In [ ]:
import zipfile

# Extract the archive
local_zip = './rps.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('tmp/rps-train')
zip_ref.close()

local_zip = './rps-test-set.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('tmp/rps-test')
zip_ref.close()

In [ ]:
import os

base_dir = 'tmp/rps-train/rps'

rock_dir = os.path.join(base_dir, 'rock')
paper_dir = os.path.join(base_dir, 'paper')
scissors_dir = os.path.join(base_dir, 'scissors')

print('total training rock images:', len(os.listdir(rock_dir)))
print('total training paper images:', len(os.listdir(paper_dir)))
print('total training scissors images:', len(os.listdir(scissors_dir)))

rock_files = os.listdir(rock_dir)
print(rock_files[:10])

paper_files = os.listdir(paper_dir)
print(paper_files[:10])

scissors_files = os.listdir(scissors_dir)
print(scissors_files[:10])

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pic_index = 2

next_rock = [os.path.join(rock_dir, fname)
                for fname in rock_files[pic_index-2:pic_index]]
next_paper = [os.path.join(paper_dir, fname)
                for fname in paper_files[pic_index-2:pic_index]]
next_scissors = [os.path.join(scissors_dir, fname)
                for fname in scissors_files[pic_index-2:pic_index]]

plt.figure(figsize=(12, 2.5))
for i, img_path in enumerate(next_rock+next_paper+next_scissors):
  img = mpimg.imread(img_path)
  plt.subplot(1, 6, i + 1)
  plt.imshow(img)
  plt.axis('Off')
plt.show()

## Build the model

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(42)

model = tf.keras.models.Sequential([
    # Note the input shape is the desired size of the image 150x150 with 3 bytes color
    # This is the first convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    # The second convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    # The third convolution
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    # The fourth convolution
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    # Flatten the results to feed into a DNN
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.5),
    # 512 neuron hidden layer
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

# Print the model summary
model.summary()

In [ ]:
# Set the training parameters
model.compile(loss = 'categorical_crossentropy', optimizer='rmsprop', metrics=['accuracy'])

## Prepare the ImageDataGenerator

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

TRAINING_DIR = "tmp/rps-train/rps"
training_datagen = ImageDataGenerator(
      rescale = 1./255,
	    rotation_range=40,
      width_shift_range=0.2,
      height_shift_range=0.2,
      shear_range=0.2,
      zoom_range=0.2,
      horizontal_flip=True,
      fill_mode='nearest')

VALIDATION_DIR = "tmp/rps-test/rps-test-set"
validation_datagen = ImageDataGenerator(rescale = 1./255)

train_generator = training_datagen.flow_from_directory(
	TRAINING_DIR,
	target_size=(150,150),
	class_mode='categorical',
  batch_size=126
)

validation_generator = validation_datagen.flow_from_directory(
	VALIDATION_DIR,
	target_size=(150,150),
	class_mode='categorical',
  batch_size=126
)
print(train_generator.class_indices)

## Train the model and evaluate the results

In [ ]:
# Train the model
history = model.fit(train_generator, epochs=25, steps_per_epoch=20, validation_data = validation_generator, verbose = 1, validation_steps=3)

In [ ]:
import matplotlib.pyplot as plt

# Plot the results
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, 'r', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend(loc=0)
plt.figure()

plt.plot(epochs, loss, 'r', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend(loc=0)
plt.show()
model.save('lab10_4_rps.keras')

## Model Prediction
Замість `files.upload()` використано варіант базового ноутбука з папкою зображень (як «CODE BLOCK FOR OLDER VERSIONS OF SAFARI»),
бо всі зображення завантажуються автоматично. Порядок класів виходу: **paper, rock, scissors**.

In [ ]:
import re, io, requests
from PIL import Image
UA = {'User-Agent': 'Mozilla/5.0 (student lab project; ChNU)'}
def fetch_images(items, folder):
    '''Завантажує зображення: items = [(ім'я_файлу, адреса сторінки pixel-shot або пряма адреса зображення)]'''
    os.makedirs(folder, exist_ok=True); ok = []
    for name, page in items:
        try:
            if 'pixel-shot.com/en/showcase' in page:
                html = requests.get(page, headers=UA, timeout=20).text
                url = re.search(r'https://img\.pixel-shot\.com/[^"\'\s]+?showcase\.(?:webp|jpg|jpeg|png)', html).group(0)
            else:
                url = page
            r = requests.get(url, headers=UA, timeout=30); r.raise_for_status()
            Image.open(io.BytesIO(r.content)).convert('RGB').save(os.path.join(folder, name + '.jpg'), quality=92)
            ok.append(name)
        except Exception as e:
            print('не вдалося завантажити', name, '-', type(e).__name__)
    print(f'{folder}: завантажено {len(ok)} з {len(items)}')
    return ok

def show_folder(folder, title, cols=7):
    fs = sorted(f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    rows = max(1, (len(fs) + cols - 1) // cols)
    plt.figure(figsize=(cols * 2.3, rows * 2.5))
    for i, f in enumerate(fs):
        plt.subplot(rows, cols, i + 1); plt.imshow(Image.open(os.path.join(folder, f))); plt.title(f, fontsize=8); plt.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.show()

import numpy as np, pandas as pd
from tensorflow.keras.utils import load_img, img_to_array
CLASSES = ['paper', 'rock', 'scissors']
UA_CLS = {'paper': 'папір', 'rock': 'камінь', 'scissors': 'ножиці'}

def to_square(img, fill=(255, 255, 255)):
    '''попередня обробка: доповнюємо білим до квадрата (як навчальні фото 300×300 на білому фоні) і зменшуємо до 150×150'''
    w, h = img.size; s = max(w, h)
    canvas = Image.new('RGB', (s, s), fill); canvas.paste(img, ((s - w) // 2, (s - h) // 2))
    return canvas.resize((150, 150))

def predict_folder(folder, title, truth_from_name=True, square=True):
    fs = sorted(f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    rows, cols = [], 6; rn = int(np.ceil(len(fs) / cols))
    plt.figure(figsize=(cols * 2.6, rn * 2.9))
    for i, f in enumerate(fs):
        img = Image.open(os.path.join(folder, f)).convert('RGB')
        img = to_square(img) if square else img.resize((150, 150))
        x = img_to_array(img) / 255.                       # нормалізація, як у навчальному генераторі
        classes = model.predict(np.expand_dims(x, axis=0), batch_size=10, verbose=0)[0]
        ans = CLASSES[int(np.argmax(classes))]
        truth = next((c for c in CLASSES if f.lower().startswith(c)), None) if truth_from_name else None
        plt.subplot(rn, cols, i + 1); plt.imshow(img); plt.axis('off')
        plt.title(f'{UA_CLS[ans]} ({classes.max()*100:.0f}%)\n{f}', fontsize=8,
                  color='black' if truth is None else ('green' if ans == truth else 'red'))
        rows.append({'файл': f, 'справжній': UA_CLS.get(truth, '?'), 'відповідь': UA_CLS[ans],
                     **{UA_CLS[c] + ', %': round(float(p) * 100, 1) for c, p in zip(CLASSES, classes)}})
    plt.suptitle(title); plt.tight_layout(); plt.show()
    df = pd.DataFrame(rows)
    if truth_from_name:
        print(f'{title}: вірно {(df["справжній"] == df["відповідь"]).sum()} з {len(df)}')
    return df

### 15. Тестові зображення з набору, наведеного в лекції (`rps-validation`) — 9 зображень, по 3 кожного класу

In [ ]:
!wget -q https://storage.googleapis.com/learning-datasets/rps-validation.zip
zipfile.ZipFile('rps-validation.zip').extractall('tmp/rps-validation')
src = 'tmp/rps-validation'
allf = sorted(f for r, _, fs in os.walk(src) for f in fs if f.endswith('.png'))
os.makedirs('/tmp/images/lecture', exist_ok=True)
for c in CLASSES:
    for f in [f for f in allf if f.startswith(c)][:3]:
        p = next(os.path.join(r, f) for r, _, fs in os.walk(src) if f in fs)
        open('/tmp/images/lecture/' + f, 'wb').write(open(p, 'rb').read())
lecture = predict_folder('/tmp/images/lecture', 'rps-validation (зображення з лекції)', square=False)
lecture

### 16. Реальні фотографії рук
Справжні фото рук з інтернету (фотобанк pixel-shot.com та Wikimedia Commons) — кулак, розкрита долоня, два пальці.
Попередня обробка: доповнення білим до квадрата і зменшення до 150×150 (навчальні зображення — 3D-руки на білому фоні 300×300).

In [ ]:
HANDS = [
 [
  "rock_1",
  "https://pixel-shot.com/en/showcase/i-6623802-0/hand-of-woman-with-clenched-fist-on-white-background"
 ],
 [
  "rock_2",
  "https://pixel-shot.com/en/showcase/i-2964436-0/raised-male-hand-with-clenched-fist-on-white-background"
 ],
 [
  "rock_3",
  "https://pixel-shot.com/en/showcase/i-6570321-0/hand-of-african-american-man-with-clenched-fist-on-white-background"
 ],
 [
  "rock_4",
  "https://commons.wikimedia.org/wiki/Special:FilePath/Rock-paper-scissors_(rock).png"
 ],
 [
  "paper_1",
  "https://pixel-shot.com/en/showcase/i-2337946-0/open-palm-hand-gesture-on-white-background"
 ],
 [
  "paper_2",
  "https://pixel-shot.com/en/showcase/i-6550857-0/male-hand-with-open-palm-on-white-background"
 ],
 [
  "paper_3",
  "https://pixel-shot.com/en/showcase/i-1312497-0/human-hand-isolated-on-white"
 ],
 [
  "paper_4",
  "https://commons.wikimedia.org/wiki/Special:FilePath/Rock-paper-scissors_(paper).png"
 ],
 [
  "scissors_1",
  "https://pixel-shot.com/en/showcase/i-5213807-0/female-hand-showing-victory-gesture-on-white-background"
 ],
 [
  "scissors_2",
  "https://pixel-shot.com/en/showcase/i-6570236-0/hand-of-african-american-man-showing-victory-gesture-on-white-background"
 ],
 [
  "scissors_3",
  "https://pixel-shot.com/en/showcase/i-6385375-0/female-hand-showing-two-fingers-on-white-background"
 ],
 [
  "scissors_4",
  "https://commons.wikimedia.org/wiki/Special:FilePath/Rock-paper-scissors_(scissors).png"
 ]
]
fetch_images(HANDS, '/tmp/images/hands')
hands = predict_folder('/tmp/images/hands', 'Реальні фото рук з інтернету')
hands

### Додатковий експеримент: справжні камені, ножиці і папір
12 фото предметів (не рук) завантажено через панель «Файли» в `/content/objects/` (`rock_*`, `scissors_*`, `paper_*`).
Мета — перевірити, на що реагує мережа: на форму руки чи на «сенс» класу.

In [ ]:
if os.path.isdir('/content/objects') and os.listdir('/content/objects'):
    objects = predict_folder('/content/objects', 'Предмети замість жестів (додатковий експеримент)')
else:
    print('Папка /content/objects порожня — експеримент пропущено')

In [ ]:
summary = pd.DataFrame([
    {'набір': 'rps-validation (лекція, 3D-руки)', 'вірно': f"{(lecture['справжній'] == lecture['відповідь']).sum()}/{len(lecture)}"},
    {'набір': 'реальні фото рук', 'вірно': f"{(hands['справжній'] == hands['відповідь']).sum()}/{len(hands)}"},
] + ([{'набір': 'предмети (камені, ножиці, папір)', 'вірно': f"{(objects['справжній'] == objects['відповідь']).sum()}/{len(objects)}"}] if 'objects' in globals() else []))
summary.to_csv('lab10_4_rps_summary.csv', index=False)
!zip -q -r lab10_4_rps_test_images.zip /tmp/images
from google.colab import files
for f in ['lab10_4_rps.keras', 'lab10_4_rps_summary.csv', 'lab10_4_rps_test_images.zip']:
    files.download(f)
summary